# Sesión 4 — MBD

## Regresión logística

Construimos un modelo de probabilidad de pago y evaluamos sus predicciones en un conjunto de test.
La respuesta es **`y = yes`: el crédito se paga**; codificamos ese suceso como 1 y `no` como 0.

En VS Code, selecciona un kernel de Python y ejecuta las celdas en orden con **Shift + Enter**.
Abre **Variables** para inspeccionar las tablas y usa **Esquema / Outline** para navegar entre apartados.
Los desplegables y deslizadores se activan al ejecutar sus celdas.

Para repetir el análisis desde cero, usa **Reiniciar** y **Ejecutar todo**.
[Guía de notebooks en VS Code](https://code.visualstudio.com/docs/datascience/jupyter-notebooks).

### Variables del conjunto de datos

| Variable | Descripción |
| --- | --- |
| `status` | Situación de la cuenta corriente |
| `duration` | Duración del crédito, en meses |
| `credit.hist` | Historial crediticio |
| `purpose` | Finalidad del crédito |
| `credit.amo` | Importe del crédito |
| `savings` | Cuenta de ahorro o bonos |
| `employed.time` | Antigüedad en el empleo actual |
| `installment` | Cuota como porcentaje de la renta disponible |
| `status.sex` | Estado civil y sexo |
| `debtors` | Otros deudores o avalistas |
| `residence.time` | Tiempo en la residencia actual |
| `property` | Propiedades |
| `age` | Edad, en años |
| `installment2` | Otros planes de pago a plazos |
| `housing` | Tipo de vivienda |
| `num.credits` | Número de créditos existentes en este banco |
| `job` | Tipo de empleo |
| `num.people` | Número de personas a cargo |
| `telephone` | Disponibilidad de teléfono |
| `foreign` | Trabajador extranjero |
| `y` | Crédito pagado: `no` o `yes` |

## Parte 1. Construir el modelo

### Preparar la sesión

La siguiente celda vacía las variables de la sesión. Después importamos las librerías necesarias.
Si falta alguna en el kernel seleccionado, descomenta y ejecuta la línea de instalación.

In [ ]:
%reset -f

In [ ]:
# %pip install numpy pandas matplotlib scipy statsmodels scikit-learn ipywidgets threadpoolctl ipykernel

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
import statsmodels.api as sm
from IPython.display import display
from scipy.optimize import brentq
from scipy.stats import chi2, norm
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.calibration import calibration_curve
from statsmodels.graphics.mosaicplot import mosaic
from threadpoolctl import threadpool_limits

%matplotlib inline
%config InteractiveShell.ast_node_interactivity = "all"
plt.rcParams.update({"figure.figsize": (8, 4), "figure.dpi": 100})
pd.set_option("display.max_columns", 25)

### Lectura de datos e inspección

Localizamos la carpeta `datos` del proyecto y leemos el conjunto de entrenamiento.
Las columnas de texto se representan como variables categóricas.

In [ ]:
for carpeta in (Path.cwd(), *Path.cwd().parents):
    if (carpeta / "datos" / "bank0_train.txt").is_file():
        DATA_PATH = carpeta / "datos"
        break
else:
    raise FileNotFoundError("Abre el notebook dentro de la carpeta LaSalle_Estadistica.")

datos = pd.read_csv(DATA_PATH / "bank0_train.txt", sep=";")
for columna in datos.select_dtypes(include="object"):
    datos[columna] = datos[columna].astype("category")

In [ ]:
datos

In [ ]:
datos.describe(include="all")

### Descriptiva bivariante

#### Variables categóricas

Identificamos los tipos y seleccionamos las variables categóricas explicativas, excluyendo la respuesta `y`.

In [ ]:
datos.dtypes

In [ ]:
var_cat = [columna for columna in datos.select_dtypes(include="category") if columna != "y"]
var_cat

#### Diagramas de mosaico

Elige una variable para explorar su relación con el pago. El área de cada rectángulo representa su frecuencia.
Las etiquetas se muestran en varias líneas cuando son largas. Las categorías con poco espacio
se consultan en la tabla de frecuencias que acompaña al gráfico.

In [ ]:
def mosaico_pago(variable):
    import textwrap
    fig, ax = plt.subplots(figsize=(11, 6))
    _, rectangulos = mosaic(datos, ["y", variable], ax=ax,
           properties=lambda clave: {"color": "#e76f51" if clave[0] == "no" else "#52b788"},
           labelizer=lambda clave: "",
           title=variable, axes_label=False)
    for clave, (x, y_rect, ancho, alto) in rectangulos.items():
        if alto >= 0.08:
            ax.text(x + ancho / 2, y_rect + alto / 2, textwrap.fill(clave[1], width=22),
                    ha="center", va="center", fontsize=9)
    centros = []
    for clase in ["no", "yes"]:
        x, y_rect, ancho, alto = next(rect for clave, rect in rectangulos.items() if clave[0] == clase)
        centros.append(x + ancho / 2)
    ax.set_xticks(centros)
    ax.set_xticklabels(["No paga", "Paga"])
    ax.set_yticks([])
    ax.set(xlabel="Crédito pagado", ylabel=variable)
    fig.tight_layout()
    plt.show()
    display(pd.crosstab(datos[variable], datos["y"]))

In [ ]:
control_mosaico = widgets.interactive(
    mosaico_pago,
    variable=widgets.Dropdown(options=var_cat, description="Variable", layout=widgets.Layout(width="400px")),
)
display(control_mosaico)

#### Densidad condicional para variables numéricas

Representamos la probabilidad de cada respuesta a lo largo de una variable numérica,
estimada mediante densidades gaussianas ponderadas por la frecuencia de cada clase.
La curva se evalúa en **512 puntos**. Cambia la variable y el suavizado para explorar el patrón.

In [ ]:
var_num = list(datos.select_dtypes(include="number").columns)
var_num

In [ ]:
def densidad_pago(variable, suavizado):
    x = datos[variable].to_numpy(dtype=float)
    escala = min(x.std(ddof=1), (np.quantile(x, 0.75) - np.quantile(x, 0.25)) / 1.34)
    if escala <= 0:
        escala = x.std(ddof=1) or 1.0
    ancho = 0.9 * escala * len(x) ** (-0.2) * suavizado
    rejilla = np.linspace(x.min() - 3 * ancho, x.max() + 3 * ancho, 512)
    densidades = []
    for clase in ["no", "yes"]:
        valores = x[(datos["y"] == clase).to_numpy()]
        kernels = norm.pdf((rejilla[:, None] - valores[None, :]) / ancho) / ancho
        densidades.append(kernels.sum(axis=1) / len(x))
    densidades = np.asarray(densidades)
    probabilidades = densidades / densidades.sum(axis=0)
    fig, ax = plt.subplots()
    ax.stackplot(rejilla, probabilidades, labels=["No paga", "Paga"], colors=["#e76f51", "#52b788"])
    ax.set(xlabel=variable, ylabel="Probabilidad condicional", title=variable, ylim=(0, 1))
    ax.legend(loc="upper right")
    fig.tight_layout()
    plt.show()

In [ ]:
control_densidad = widgets.interactive(
    densidad_pago,
    variable=widgets.Dropdown(options=var_num, description="Variable"),
    suavizado=widgets.FloatSlider(value=1, min=0.4, max=2, step=0.1,
                                 description="Suavizado", continuous_update=False),
)
display(control_densidad)

### Categorías de referencia

Elegimos las referencias por su interpretación. Cada coeficiente categórico describe el cambio
respecto a la categoría de referencia, manteniendo constantes las demás variables.
Colocamos esa categoría en primer lugar antes de generar las variables indicadoras.

In [ ]:
referencias = {
    "status": "no checking account",
    "credit.hist": "no credits taken/all credits paid back duly",
    "purpose": "others",
    "savings": "unknown/no savings account",
    "employed.time": "unemployed",
    "status.sex": "male:single",
    "debtors": "none",
    "property": "unknown/no property",
    "installment2": "none",
    "housing": "for free",
    "job": "unskilled - resident",
    "telephone": "none",
    "foreign": "no",
}

for columna, referencia in referencias.items():
    categorias = list(datos[columna].cat.categories)
    datos[columna] = datos[columna].cat.reorder_categories(
        [referencia] + [categoria for categoria in categorias if categoria != referencia]
    )

pd.Series(referencias, name="Categoría de referencia").to_frame()

### Estimar el modelo completo

Usamos una regresión logística binomial. `y` vale 1 cuando el crédito se paga.
`drop_first=True` omite la categoría de referencia de cada predictor categórico.
La columna `const` representa el intercepto.

In [ ]:
y = (datos["y"] == "yes").astype(int)
predictores = [columna for columna in datos.columns if columna != "y"]
X = pd.get_dummies(datos[predictores], drop_first=True, prefix_sep="=", dtype=float)
X = sm.add_constant(X, has_constant="add")

grupos = {
    variable: [columna for columna in X if columna == variable or columna.startswith(variable + "=")]
    for variable in predictores
}

In [ ]:
def ajustar_modelo(variables):
    columnas = ["const"] + [columna for variable in variables for columna in grupos[variable]]
    with threadpool_limits(limits=1):
        return sm.GLM(y, X[columnas], family=sm.families.Binomial(), missing="raise").fit(maxiter=100)

In [ ]:
mod_glm0 = ajustar_modelo(predictores)
mod_glm0.summary()

### Selección automática por AIC

#### Valores ausentes

Todos los modelos candidatos deben usar las mismas observaciones. Comprobamos que no haya valores ausentes.

In [ ]:
datos.isna().sum()

#### Selección hacia atrás

Partimos del modelo completo. En cada paso eliminamos la variable cuya retirada reduce más el AIC.
Una variable categórica entra o sale con todas sus categorías. Nos detenemos cuando ninguna eliminación mejora el AIC.

In [ ]:
def seleccion_atras(variables):
    if datos.isna().any().any():
        raise ValueError("Resuelve los valores ausentes antes de seleccionar variables.")
    activas = list(variables)
    modelo = ajustar_modelo(activas)
    historial = [{"Paso": 0, "Eliminada": "—", "AIC": modelo.aic, "Variables": len(activas)}]
    while activas:
        candidatos = []
        for variable in activas:
            restantes = [nombre for nombre in activas if nombre != variable]
            candidato = ajustar_modelo(restantes)
            candidatos.append((candidato.aic, variable, candidato))
        aic, retirada, candidato = min(candidatos, key=lambda resultado: resultado[0])
        if aic >= modelo.aic - 1e-7:
            break
        activas.remove(retirada)
        modelo = candidato
        historial.append({"Paso": len(historial), "Eliminada": retirada,
                          "AIC": aic, "Variables": len(activas)})
    return modelo, activas, pd.DataFrame(historial)

In [ ]:
mod_glm1, variables_seleccionadas, pasos_aic = seleccion_atras(predictores)
pasos_aic

In [ ]:
variables_seleccionadas
mod_glm1.summary()

#### Importancia según la significación estadística

Ordenamos los coeficientes por el valor absoluto de su estadístico z, excluyendo el intercepto.
Este criterio describe evidencia estadística; no mide por sí solo el tamaño del efecto.

In [ ]:
importancia = pd.DataFrame({
    "Importancia (|z|)": mod_glm1.tvalues.drop("const").abs(),
    "p-valor": mod_glm1.pvalues.drop("const"),
}).sort_values("Importancia (|z|)", ascending=False)
importancia

### Validación

#### Pagos esperados y observados por grupos de probabilidad

Agrupamos las probabilidades predichas por cuantiles. En cada grupo sumamos los pagos observados
y las probabilidades de pago esperadas. Incluimos todas las observaciones, también las de probabilidad mínima.

In [ ]:
pr = mod_glm1.predict(X[mod_glm1.model.exog_names])

def tabla_por_cuantiles(observado, predicho, grupos=10):
    tabla = pd.DataFrame({"observado": np.asarray(observado), "predicho": np.asarray(predicho)})
    tabla["intervalo"] = pd.qcut(tabla["predicho"], q=grupos, duplicates="drop")
    resumen = tabla.groupby("intervalo", observed=True).agg(
        n=("observado", "size"), obs=("observado", "sum"), exp=("predicho", "sum"),
    )
    return resumen

validacion = tabla_por_cuantiles(y, pr)
validacion

In [ ]:
def pagos_por_intervalo(grupos):
    tabla = tabla_por_cuantiles(y, pr, grupos)
    posiciones = np.arange(1, len(tabla) + 1)
    fig, ax = plt.subplots()
    ax.vlines(posiciones + 0.05, 0, tabla["exp"], color="black", linewidth=2, label="Pagan: esperados")
    ax.vlines(posiciones - 0.05, 0, tabla["obs"], color="red", linewidth=2, label="Pagan: observados")
    ax.set(xlabel="Intervalos", ylabel="Frecuencias", xticks=posiciones)
    ax.legend(loc="upper left")
    plt.show()

In [ ]:
control_validacion = widgets.interactive(
    pagos_por_intervalo,
    grupos=widgets.IntSlider(value=10, min=5, max=15, description="Grupos", continuous_update=False),
)
display(control_validacion)

#### Test de Hosmer–Lemeshow

El test contrasta la concordancia entre frecuencias observadas y esperadas.
Un p-valor inferior a 0.05 aporta evidencia de falta de ajuste; un valor mayor no demuestra que el modelo sea correcto.
Usamos 10 grupos de probabilidades y grados de libertad iguales al número de grupos menos 2.

In [ ]:
def hosmer_lemeshow(observado, predicho, grupos=10):
    tabla = tabla_por_cuantiles(observado, predicho, grupos)
    if len(tabla) <= 2:
        raise ValueError("Se necesitan más de dos grupos de probabilidad distintos.")
    no_obs = tabla["n"] - tabla["obs"]
    no_exp = tabla["n"] - tabla["exp"]
    if (tabla["exp"] <= 0).any() or (no_exp <= 0).any():
        raise ValueError("Hay grupos con frecuencia esperada cero.")
    estadistico = (((tabla["obs"] - tabla["exp"]) ** 2 / tabla["exp"])
                   + ((no_obs - no_exp) ** 2 / no_exp)).sum()
    gl = len(tabla) - 2
    return pd.Series({"Chi-cuadrado": estadistico, "Grados de libertad": gl,
                      "p-valor": chi2.sf(estadistico, gl)})

In [ ]:
hosmer_lemeshow(y, pr)

### Estimación de un odds ratio

Los *odds* de pago son `p / (1 - p)`. El odds ratio (OR) multiplica esos odds, manteniendo constantes
los demás predictores. No representa una multiplicación directa de la probabilidad.

#### Variable categórica

`foreign=yes` se interpreta respecto a la referencia `foreign=no`.

In [ ]:
np.exp(mod_glm1.params["foreign=yes"])

#### Variable numérica: cambio en los odds por cada año adicional de edad

In [ ]:
np.exp(mod_glm1.params["age"])

#### Intervalos de confianza

Calculamos intervalos del 95 % mediante el perfil de verosimilitud y exponenciamos sus límites para obtener
los intervalos de los OR. La función fija un coeficiente y vuelve a estimar los demás hasta encontrar cada límite.
Esta celda puede tardar unos segundos.

In [ ]:
def intervalos_perfil(modelo, nivel=0.95):
    matriz = np.asarray(modelo.model.exog, dtype=float)
    respuesta = np.asarray(modelo.model.endog, dtype=float)
    beta = np.asarray(modelo.params)
    errores = np.asarray(modelo.bse)
    corte = chi2.ppf(nivel, df=1)
    limites = []
    with threadpool_limits(limits=1):
        for j, centro in enumerate(beta):
            otras = np.arange(len(beta)) != j

            def diferencia(valor):
                ajuste = sm.GLM(respuesta, matriz[:, otras], offset=matriz[:, j] * valor,
                                family=sm.families.Binomial()).fit(maxiter=100, tol=1e-9)
                if not ajuste.converged or not np.isfinite(ajuste.llf):
                    raise RuntimeError(f"El perfil no converge para {modelo.params.index[j]}.")
                return 2 * (modelo.llf - ajuste.llf) - corte

            extremos = []
            for signo in [-1, 1]:
                paso = max(2 * errores[j], 1e-8)
                extremo = centro + signo * paso
                for intento in range(30):
                    if diferencia(extremo) >= 0:
                        break
                    paso *= 2
                    extremo = centro + signo * paso
                else:
                    raise RuntimeError("No se encontró un límite finito para el perfil.")
                inferior, superior = sorted([centro, extremo])
                extremos.append(brentq(diferencia, inferior, superior, xtol=1e-10))
            limites.append(extremos)
    return pd.DataFrame(limites, index=modelo.params.index, columns=["2.5 %", "97.5 %"])

In [ ]:
IC = intervalos_perfil(mod_glm1)
np.exp(IC).round(2)

### Estimación de la probabilidad de pago

#### Probabilidades predichas

Cada valor de `pr` es una probabilidad entre 0 y 1.

In [ ]:
pr = mod_glm1.predict(X[mod_glm1.model.exog_names])
pr

#### Observación con la mayor probabilidad de pago

In [ ]:
pos_max = pr.idxmax()
pr.loc[pos_max]
datos.loc[pos_max, "y"]

#### Observación con la menor probabilidad de pago

In [ ]:
pos_min = pr.idxmin()
pr.loc[pos_min]
datos.loc[pos_min, "y"]

#### Distribución de las probabilidades según el pago observado

In [ ]:
def cajas_probabilidad(probabilidades, respuesta, titulo):
    fig, ax = plt.subplots()
    grupos = [np.asarray(probabilidades)[np.asarray(respuesta) == clase] for clase in ["no", "yes"]]
    ax.boxplot(grupos)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(["No paga", "Paga"])
    ax.set(ylabel="Probabilidad predicha de pago", ylim=(0, 1), title=titulo)
    plt.show()

cajas_probabilidad(pr, datos["y"], "Entrenamiento")

### Curva ROC y AUC

La curva representa la sensibilidad frente a `1 − especificidad` al variar el punto de corte.
El AUC resume la capacidad de ordenar los pagos por encima de los impagos.

In [ ]:
def dibujar_roc(observado, probabilidades, titulo):
    fpr, tpr, cortes = roc_curve(observado, probabilidades)
    area = roc_auc_score(observado, probabilidades)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"AUC = {area:.3f}")
    ax.plot([0, 1], [0, 1], "--", color="grey")
    ax.set(xlabel="1 − Especificidad", ylabel="Sensibilidad", title=titulo,
           xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="lower right")
    plt.show()

dibujar_roc(y, pr, "Curva ROC — entrenamiento")

In [ ]:
roc_auc_score(y, pr)

### Calibración del modelo

Agrupamos las predicciones en intervalos de igual amplitud. En cada grupo representamos la probabilidad media
predicha y la proporción real de pagos. La diagonal indica concordancia perfecta.
El número de intervalos se puede modificar con el control.

In [ ]:
df_calibra = pd.DataFrame({"plotID": np.arange(1, len(datos) + 1), "Observed": y, "Predicted1": pr})
df_calibra.head()

In [ ]:
def calibracion_modelo(intervalos):
    observadas, predichas = calibration_curve(
        df_calibra["Observed"], df_calibra["Predicted1"], n_bins=intervalos, strategy="uniform",
    )
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(predichas, observadas, "o-", label="Modelo")
    ax.plot([0, 1], [0, 1], "--", color="grey", label="Calibración perfecta")
    ax.set(xlabel="Probabilidad predicha", ylabel="Probabilidad observada",
           title="Calibración del modelo", xlim=(0, 1), ylim=(0, 1))
    ax.legend()
    plt.show()

In [ ]:
control_calibracion = widgets.interactive(
    calibracion_modelo,
    intervalos=widgets.IntSlider(value=10, min=5, max=20, description="Intervalos", continuous_update=False),
)
display(control_calibracion)

## Parte 2. Evaluar los resultados

### Leer los datos de test

Aplicamos las mismas categorías y referencias del entrenamiento.
El modelo se evalúa con estos datos, sin volver a estimar sus coeficientes.

In [ ]:
test = pd.read_csv(DATA_PATH / "bank0_test.txt", sep=";")
for columna in datos.select_dtypes(include="category"):
    desconocidas = set(test[columna].dropna()) - set(datos[columna].cat.categories)
    if desconocidas:
        raise ValueError(f"Categorías nuevas en {columna}: {desconocidas}")
    test[columna] = pd.Categorical(test[columna], categories=datos[columna].cat.categories)

X_test = pd.get_dummies(test[predictores], drop_first=True, prefix_sep="=", dtype=float)
X_test = sm.add_constant(X_test, has_constant="add").reindex(columns=X.columns, fill_value=0)
y_test = (test["y"] == "yes").astype(int)

### Predicciones y valores reales

In [ ]:
pr_test = mod_glm1.predict(X_test[mod_glm1.model.exog_names])
pr_test

In [ ]:
cajas_probabilidad(pr_test, test["y"], "Test")

In [ ]:
dibujar_roc(y_test, pr_test, "Curva ROC — test")

In [ ]:
roc_auc_score(y_test, pr_test)

### Sensibilidad, especificidad y exactitud por punto de corte

Predecimos pago cuando la probabilidad es **mayor** que el punto de corte.

- **Sensibilidad**: proporción de personas que pagan clasificadas como pago.
- **Especificidad**: proporción de personas que no pagan clasificadas como impago.
- **Exactitud**: proporción total de clasificaciones correctas.

In [ ]:
def metricas_corte(observado, probabilidades, corte):
    real = np.asarray(observado, dtype=bool)
    predicho = np.asarray(probabilidades) > corte
    return {"cutpoints": corte,
            "sens": (predicho & real).sum() / real.sum(),
            "esp": (~predicho & ~real).sum() / (~real).sum(),
            "acc": (predicho == real).mean()}

In [ ]:
cutpoints = np.unique(np.r_[0.0, pr_test, 1.0])
df = pd.DataFrame([metricas_corte(y_test, pr_test, corte) for corte in cutpoints])
df.round(3)

### Elegir un punto de corte: matriz de confusión

Con un umbral de 0.5, concedemos crédito a las observaciones con probabilidad de pago superior a 0.5.
Las filas representan la decisión de conceder crédito y las columnas el pago real.

In [ ]:
test["doy_credito"] = np.where(pr_test > 0.5, "si", "no")
pd.crosstab(test["doy_credito"], test["y"])

In [ ]:
(100 * pd.crosstab(test["doy_credito"], test["y"], normalize="index")).round(1)

#### Explorar el umbral

Mueve el punto de corte. Se actualizan la decisión `test["doy_credito"]`, la matriz de confusión,
los porcentajes por fila y las métricas. El punto rojo sitúa la decisión en la curva ROC.

In [ ]:
def evaluar_umbral(umbral):
    test["doy_credito"] = np.where(pr_test > umbral, "si", "no")
    matriz = pd.crosstab(test["doy_credito"], test["y"]).reindex(
        index=["no", "si"], columns=["no", "yes"], fill_value=0,
    )
    porcentajes = matriz.div(matriz.sum(axis=1).replace(0, np.nan), axis=0).mul(100).round(1)
    metricas = metricas_corte(y_test, pr_test, umbral)
    print("Matriz de confusión: filas = decisión; columnas = pago real")
    display(matriz)
    print("Porcentajes por fila (una fila vacía no tiene porcentaje definido)")
    display(porcentajes)
    display(pd.Series(metricas).rename({"cutpoints": "Umbral", "sens": "Sensibilidad",
                                       "esp": "Especificidad", "acc": "Exactitud"}).round(3))
    fpr, tpr, cortes = roc_curve(y_test, pr_test)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, pr_test):.3f}")
    ax.plot([0, 1], [0, 1], "--", color="grey")
    ax.scatter(1 - metricas["esp"], metricas["sens"], color="red", zorder=3, label=f"Umbral = {umbral:.2f}")
    ax.set(xlabel="1 − Especificidad", ylabel="Sensibilidad", xlim=(0, 1), ylim=(0, 1))
    ax.legend(loc="lower right")
    plt.show()

In [ ]:
control_umbral = widgets.interactive(
    evaluar_umbral,
    umbral=widgets.FloatSlider(value=0.5, min=0, max=1, step=0.01,
                              description="Umbral", continuous_update=False),
)
display(control_umbral)

### Documentación de las herramientas

- [Modelos lineales generalizados en statsmodels](https://www.statsmodels.org/stable/glm.html).
- [Curva ROC en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.roc_curve.html).
- [Calibración de probabilidades en scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.calibration.calibration_curve.html).